In [11]:
# Hamza Faheem DS-B,  AI Assignment #2

In [12]:
import random
import copy

In [13]:
class Card:
    def __init__(self,color,val):
     self.color = color
     self.val = val
        
    def represent(self):
     return ((self.color) + " " + str(self.val))
        
    def __equal__(self,other):
     if not isinstance(other, Card): 
       return False
     return self.color == other.color and self.value == other.value

def generateDeck():
    colors = ['Red', 'Blue', 'Green', 'Yellow']
    values = [str(i) for i in range(10)] + ['Skip']

    for c in colors:
        for v in values:
            new_card = Card(c, v)
            deck.append(new_card)
            
    random.shuffle(deck)
    return deck        

def getValidMoves():    
    validMs = []
    for card in hand:
        if card.color == top_card.color or card.val == top_card.val:
            validMs.append(card)
    return validMs

In [14]:
class gameState:
    def __init__(self, p1c, p2c, p3c, deck, top, currTurn = 1):
      self.units = {1 : p1c, 2 : p2c, 3 : p3c}  
      self.top = top
      self.deck = deck
      self.currTurn = currTurn

    def clone(self):
      return copy.deepcopy(self)

    def apply(self, P_id, cardsPlayed = "None"):
      nextTurn = (P_id % 3) + 1
      if cardsPlayed:
        self.units[P_id].remove(cardsPlayed)
        self.deck.append(self.top_card) 
        self.top_card = cardsPlayed
            
        if cardsPlayed.value == 'Skip':
            nextTurn = (nextTurn % 3) + 1
      else:
        if self.deck:
            drawnC = self.deck.pop(0)
            self.units[P_id].append(drawnC)
                
      self.currTurn = nextTurn
      return nextTurn

In [15]:
def evaluate(state, P_id, strat = "baseline"):
    myHands = state.units[P_id]
    oppsHands = [state.units[p] for p in [1, 2, 3] if p != P_id]
    
    c_ai = len(myHands)
    c_opp_avg = sum(len(hand) for hand in oppsHands) / 2
    s_cards = sum(1 for card in myHands if card.val == 'Skip')
    
    if c_ai == 0: return 1000  
    if any(len(h) == 0 for h in oppsHands): return -1000 
    
    if strat == "defensive":
        score = 50 - 4 * c_ai + 5 * c_opp_avg + 6 * s_cards
    elif strat == "offensive":
        score = 50 - 8 * c_ai + 2 * c_opp_avg + 2 * s_cards
    else:
        score = 50 - 5 * c_ai + 2 * c_opp_avg + 3 * s_cards
        
    return score

In [1]:
def minmax(state, depth, maximizing_PID):
    if depth == 0 or len(state.units[1]) == 0 or len(state.units[2]) == 0 or len(state.units[3]) == 0:
        return evaluate(state, maximizing_PID, "defensive"), None

    curr_P = state.currTurn
    valid_moves = getValidMoves(state.units[curr_P], state.top)
    
    if len(valid_moves) == 0:
        possible_moves = [None]
    else:
        possible_moves = valid_moves

    if curr_P == maximizing_PID:
        max_eval = float('-inf')
        best_move = None
        
        for move in possible_moves:
            simulated_state = state.clone()
            simulated_state.apply(curr_P, move)
            evaluatedScore, _ = minimax(simulated_state, depth - 1, maximizing_PID)
            
            if evaluatedScore > max_eval:
                max_eval = evaluatedScore
                best_move = move
                
        return max_eval, best_move

    else:
        min_eval = float('inf')
        best_move = None
        
        for move in possible_moves:
            simulated_state = state.clone()
            simulated_state.apply(curr_P, move)
            evaluatedScore, _ = minimax(simulated_state, depth - 1, maximizing_PID)
            
            if evaluatedScore < min_eval:
                min_eval = evaluatedScore
                best_move = move
                
        return min_eval, best_move

In [ ]:
def expectimax(state, depth, AI_iden):
    if depth == 0 or len(state.units[1]) == 0 or len(state.units[2]) == 0 or len(state.units[3]) == 0:
        return evaluate(state, AI_iden, "defensive"), None

    currP = state.currTurn
    valid_moves = getValidMoves(state.hands[currP], state.top)
    if currP == AI_iden:
        if len(valid_moves) > 0:
            max_eval = float('-inf')
            best_move = None
            
            for move in valid_moves:
                sim_state = state.clone()
                sim_state.apply(currP, move)
                evaluatedScore, _ = expectimax(sim_state, depth - 1, AI_iden)
                
                if evaluatedScore > max_eval:
                    max_eval = evaluatedScore
                    best_move = move
            return max_eval, best_move
            
        else:
            if len(state.deck) == 0:
                return evaluate(state, AI_iden, "offensive"), None
                
            expectedVal = 0
            prob = 1.0 / len(state.deck)
            
            for i in range(len(state.deck)):
                sim_state = state.clone()
                drawn_card = sim_state.deck.pop(i)
                sim_state.hands[currP].append(drawn_card)
                sim_state.currTurn = (currP % 3) + 1 
                
                evaluatedScore, _ = expectimax(sim_state, depth - 1, AI_iden)
                expectedVal += prob * evaluatedScore
                
            return expectedVal, None 

    else:
        if len(valid_moves) == 0:
            possible_moves = [None]
        else:
            possible_moves = valid_moves
            
        expectedVal = 0
        prob = 1.0 / len(possible_moves)
        
        for move in possible_moves:
            sim_state = state.clone()
            sim_state.apply(currP, move)
            evaluatedScore, _ = expectimax(sim_state, depth - 1, AI_iden)
            expectedVal += prob * evaluatedScore
            
        return expectedVal, random.choice(possible_moves)    